---
title: "Part 19: Data Validation with Pydantic"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/02-dev-tools/07-pydantic-validation.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/02-dev-tools/07-pydantic-validation.ipynb)

**DS-MLOps Dev Tools**

**Python 3.12+ | Author: Anthony Faustine**

## Before you begin

This notebook assumes you have completed [Part 15: Type Annotations](03-type-annotations.ipynb). Pydantic is type annotations put to work: the type hints you write in Part 15 become runtime validators that reject wrong inputs before they corrupt a pipeline.

The `grade-predictor` project continues here: a `GradeConfig` model replaces the ad-hoc defaults in `compute_grade`, and a `StudentRecord` model validates incoming data before it ever reaches a computation function.

> Callout markers used throughout this notebook are explained on the [book cover page](../../index.qmd#callout-guide).

## Learning Objectives

By the end of Part 19 you will be able to:

| # | Skill | Covered in |
| --- | --- | --- |
| 1 | Define Pydantic models and understand how they differ from dataclasses | Sec. 1 |
| 2 | Validate input data and handle `ValidationError` cleanly | Sec. 2 |
| 3 | Write field validators and cross-field validators | Sec. 3 |
| 4 | Build CLI tools with argparse and typer, and understand when to use each | Sec. 4 |
| 5 | Use `BaseSettings` to manage configuration from environment variables | Sec. 5 |
| 6 | Build a typed config object for a DS pipeline | Sec. 6 |
| 7 | Validate a batch of student records and collect all errors at once | Sec. 7 |

## 0. The Bug That Type Annotations Cannot Catch

You have just finished building the `grade-predictor` pipeline. It reads a CSV, computes weighted scores, applies a pass threshold, and writes results to a file. You add type annotations everywhere. Your type checker passes clean. You deploy.

Two weeks later, a bug report arrives: the pipeline produced a passing grade for a student whose `midterm_score` was logged as `150.0` — which is impossible on a 100-point scale. The pipeline did not crash. The math ran correctly on the wrong number. A downstream report was wrong, and nobody noticed until a student appealed their grade.

The problem was not the computation. It was that nothing stopped the invalid value from entering the pipeline in the first place. Type annotations say *what type a value should be*. They do not say *whether a value of that type makes sense at runtime*. `midterm_score: float` accepts any float: `150.0`, `-20.0`, `float("inf")`. The annotation is a hint to the reader and the type checker. It is not a gate.

**Pydantic** ([pydantic.dev](https://docs.pydantic.dev/latest/)) turns annotations into gates. Define a model, describe the constraints, and Pydantic enforces them on every construction — before the value reaches any computation. One validation point at entry; everything downstream trusts the types.

### Install

Pydantic is in `pyproject.toml`. For a standalone project:

```bash
uv add pydantic pydantic-settings    # or: pip install pydantic pydantic-settings
```

A Python dataclass accepts any value for any field, regardless of what the annotation says. A Pydantic model validates on construction. Pass the wrong type and it raises an error immediately, before the value reaches any computation:

In [ ]:
from dataclasses import dataclass

from pydantic import BaseModel


@dataclass
class DataclassStudent:
    student_id: str
    midterm_score: float


class PydanticStudent(BaseModel):
    student_id: str
    midterm_score: float


# Dataclass: accepts "eighty" silently
dc = DataclassStudent(student_id="S0001", midterm_score="eighty")
print(dc.midterm_score)  # "eighty" -- no error

# Pydantic: raises ValidationError immediately
try:
    ps = PydanticStudent(student_id="S0001", midterm_score="eighty")
except Exception as e:
    print(type(e).__name__, e)

Pydantic also coerces where the conversion is unambiguous. `"85"` becomes `85.0` for a `float` field; `"true"` becomes `True` for a `bool` field. This makes it practical for inputs from CSV files, APIs, or environment variables, where everything arrives as a string.

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Validate at the boundary, not inside the pipeline</span><br><br>
A pipeline that validates its inputs at the entry point can trust every value it works with from that point on. A pipeline that validates inside individual functions repeats the same checks everywhere and still misses values that enter through unexpected paths. Pydantic at the boundary is the architectural principle; the model is the mechanism.
</div>

## 2. `StudentRecord`: Validating Incoming Data

Define a model for a student record coming in from a CSV or API. The `model_config` attribute controls Pydantic v2's behaviour:

In [ ]:
import re
from typing import Annotated

from pydantic import BaseModel, Field, ValidationError, field_validator


class StudentRecord(BaseModel):
    model_config = {"str_strip_whitespace": True, "validate_assignment": True}

    student_id: str
    midterm_score: Annotated[float, Field(ge=0.0, le=100.0)]
    final_score: Annotated[float, Field(ge=0.0, le=100.0)]
    project_score: Annotated[float, Field(ge=0.0, le=100.0)]
    program: str
    has_internet: bool = True

    @field_validator("student_id")
    @classmethod
    def student_id_format(cls, v: str) -> str:
        if not re.match(r"^S\d{4}$", v):
            raise ValueError(f"student_id must match S followed by 4 digits, got '{v}'")
        return v

`Field(ge=0.0, le=100.0)` means "greater than or equal to 0, less than or equal to 100". Pydantic v2 uses `Annotated` with `Field` for constraints; the old `validator` decorator is replaced by `field_validator`.

Try it:

In [ ]:
# Valid record: succeeds
record = StudentRecord(
    student_id="S0042",
    midterm_score=78.5,
    final_score=82.0,
    project_score="91.0",  # string coerced to float
    program="CS",
)
print(record.model_dump())

# Invalid: midterm over 100
try:
    bad = StudentRecord(
        student_id="S0042",
        midterm_score=150.0,  # violates le=100.0
        final_score=80.0,
        project_score=75.0,
        program="CS",
    )
except ValidationError as e:
    print(e)

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 1 - Extend StudentRecord</span><br><br>
<b>Goal:</b> Add a <code>semester</code> field to <code>StudentRecord</code> that must be one of <code>"Fall"</code>, <code>"Spring"</code>, <code>"Summer"</code>. Use <code>Literal["Fall", "Spring", "Summer"]</code> as the type annotation. Try creating a record with <code>semester="Winter"</code> and confirm it raises a <code>ValidationError</code>.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>from typing import Literal

class StudentRecord(BaseModel):
    ...
    semester: Literal["Fall", "Spring", "Summer"]

StudentRecord(..., semester="Winter")  # should raise ValidationError</pre>
</div>

## 3. Field Validators and Cross-Field Validators

A `@field_validator` runs on a single field after type coercion. Use it for business rules that go beyond a simple range check.

A `@model_validator` runs after all fields are set and has access to the complete model. Use it for cross-field constraints — "weights must sum to 1.0":

In [ ]:
from typing import Annotated

from pydantic import BaseModel, Field, ValidationError, field_validator, model_validator


class GradeConfig(BaseModel):
    midterm_weight: Annotated[float, Field(gt=0.0, lt=1.0)] = 0.30
    final_weight: Annotated[float, Field(gt=0.0, lt=1.0)] = 0.45
    project_weight: Annotated[float, Field(gt=0.0, lt=1.0)] = 0.25
    pass_threshold: Annotated[float, Field(ge=50.0, le=80.0)] = 60.0

    @model_validator(mode="after")
    def weights_must_sum_to_one(self) -> "GradeConfig":
        total = self.midterm_weight + self.final_weight + self.project_weight
        if abs(total - 1.0) > 1e-6:
            raise ValueError(
                f"Weights must sum to 1.0, got {total:.6f}. "
                f"Adjust midterm ({self.midterm_weight}), "
                f"final ({self.final_weight}), or project ({self.project_weight})."
            )
        return self

The `mode="after"` argument means the validator runs after Pydantic has already validated and coerced every individual field. Use `mode="before"` when you need to transform raw input before type coercion.

In [ ]:
# Valid: weights sum to 1.0
cfg = GradeConfig(midterm_weight=0.3, final_weight=0.5, project_weight=0.2)
print(cfg)

# Invalid: weights sum to 1.05
try:
    bad_cfg = GradeConfig(midterm_weight=0.4, final_weight=0.45, project_weight=0.2)
except ValidationError as e:
    print(e)

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Use <code>model_dump()</code> and <code>model_validate()</code> to round-trip through dicts and JSON</span><br><br>
<code>config.model_dump()</code> converts a Pydantic model to a plain Python dict. <code>GradeConfig.model_validate(some_dict)</code> validates and constructs a model from a dict. <code>GradeConfig.model_validate_json(json_string)</code> parses and validates from a JSON string. These three methods cover the most common I/O patterns for config files, API payloads, and database rows.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 2 - Config from a Dict</span><br><br>
<b>Goal:</b> Create a <code>GradeConfig</code> from a Python dict using <code>model_validate()</code>. Then call <code>model_dump()</code> to get a plain dict back. Confirm the two dicts have the same values. Also confirm that a dict with weights that do not sum to 1.0 raises a <code>ValidationError</code>.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>raw = {"midterm_weight": 0.3, "final_weight": 0.45, "project_weight": 0.25}
cfg = GradeConfig.model_validate(raw)
assert cfg.model_dump() == {**raw, "pass_threshold": 60.0}</pre>
</div>

## 4. CLI Interfaces: argparse and typer

Every ML pipeline eventually needs to be run from the command line: `python train.py --data-path data/train.csv --epochs 50 --threshold 0.65`. Two tools exist for this: `argparse` (stdlib, the traditional approach) and `typer` (modern, annotation-based, by the same author as FastAPI).

### argparse: the baseline

`argparse` requires you to declare each argument manually, convert strings to types yourself, and write help text by hand:

In [ ]:
import argparse

# argparse: explicit, verbose, manual type conversion
parser = argparse.ArgumentParser(description="Grade predictor CLI")
parser.add_argument("--data-path", type=str, default="data/university_analytics.csv")
parser.add_argument("--pass-threshold", type=float, default=60.0)
parser.add_argument("--debug", action="store_true", default=False)

# In a script: args = parser.parse_args()
# In a notebook: simulate with a list
args = parser.parse_args(["--data-path", "data/train.csv", "--pass-threshold", "65.0"])
print(args.data_path, args.pass_threshold, args.debug)

### typer: annotations as the CLI spec

`typer` reads your function's type annotations and builds the parser for you. The same type hints that describe the function's signature generate `--help`, automatic type coercion, and validation:

In [ ]:
# pip install typer  /  uv add typer
# This is a module-level demo -- run as: python script.py --data-path ... from terminal

import typer

app = typer.Typer()


@app.command()
def train(
    data_path: str = typer.Option("data/university_analytics.csv", help="Path to CSV"),
    pass_threshold: float = typer.Option(60.0, min=50.0, max=80.0, help="Pass threshold"),
    debug: bool = typer.Option(False, help="Enable debug logging"),
) -> None:
    typer.echo(f"Loading data from: {data_path}")
    typer.echo(f"Pass threshold: {pass_threshold}")
    typer.echo(f"Debug: {debug}")


# In a real script: if __name__ == "__main__": app()
# Running `python train.py --help` auto-generates full usage docs.

### typer + pydantic-settings: the DS/MLOps pattern

In production, you typically want *two* sources of configuration: environment variables (for secrets, deployment targets) and CLI arguments (for per-run overrides). The pattern is:

- `pydantic-settings` reads from `.env` and environment — covered in Sec 5
- `typer` provides the CLI surface
- The typer command constructs a `PipelineConfig` from both sources

The two tools compose naturally because they share the same type-annotation vocabulary.

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: argparse vs typer</span><br><br>
Use <code>argparse</code> when you want zero dependencies and a simple script. Use <code>typer</code> when you want auto-generated help, validation, subcommands, and code that reads like the function signature it already is. For DS/MLOps tools that are both importable as a library and runnable as a CLI, typer is the better fit.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 3 - typer Command</span><br><br>
<b>Goal:</b> Write a typer command <code>validate_data</code> that accepts <code>--data-path</code> (str) and <code>--max-errors</code> (int, default 10). The command should print the arguments it received. Confirm that passing <code>--max-errors abc</code> would raise a typer error (type mismatch).
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>@app.command()
def validate_data(
    data_path: str = typer.Option(..., help="Path to CSV to validate"),
    max_errors: int = typer.Option(10, help="Stop after this many errors"),
) -> None:
    typer.echo(f"Validating {data_path} (max errors: {max_errors})")</pre>
</div>

## 5. `BaseSettings`: Config from Environment Variables

`pydantic-settings` extends Pydantic with a `BaseSettings` class that reads values from environment variables and `.env` files. This replaces the manual `os.getenv` pattern with a typed, validated config object.

```bash
uv add pydantic-settings
```

In [ ]:
# grade_predictor/config.py
from typing import Annotated

from pydantic import Field
from pydantic_settings import BaseSettings, SettingsConfigDict


class GradeSettings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=".env",
        env_prefix="GRADE_",  # GRADE_API_KEY maps to api_key
        case_sensitive=False,
    )

    api_key: str = Field(default="dev-key", description="API key for the university data service")
    data_path: str = Field(default="data/university_analytics.csv")
    pass_threshold: Annotated[float, Field(ge=50.0, le=80.0)] = 60.0
    debug: bool = False


# Load (reads .env if it exists, otherwise uses defaults)
settings = GradeSettings()
print(settings.data_path)
print(settings.pass_threshold)

With a `.env` file:

```bash
GRADE_API_KEY=secret-key-here
GRADE_DATA_PATH=data/production.csv
GRADE_PASS_THRESHOLD=65.0
```

Values from the `.env` file override defaults; environment variables override both.

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Settings validation catches misconfiguration at startup, not mid-run</span><br><br>
Without validation, a missing or malformed environment variable is discovered when the code first uses it: halfway through a pipeline run, after 20 minutes of processing. <code>BaseSettings</code> validates all settings at construction time, failing fast at startup with a clear error that lists every missing or invalid variable.
</div>

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Constructing settings inside a function that runs in a loop</span><br><br>
<code>GradeSettings()</code> reads the <code>.env</code> file and environment on every call. Calling it inside a loop that processes thousands of rows reads and validates the config thousands of times. Construct settings once at module or application level and pass the object through.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 4 - Settings with a Test Override</span><br><br>
<b>Goal:</b> Create a <code>GradeSettings</code> instance, overriding individual settings without a <code>.env</code> file by passing values directly to the constructor: <code>GradeSettings(api_key="test-key", pass_threshold=70.0)</code>. Confirm the values are set correctly and that an invalid <code>pass_threshold</code> (e.g., <code>95.0</code>) raises a <code>ValidationError</code>.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>test_settings = GradeSettings(api_key="test-key", pass_threshold=70.0)
assert test_settings.pass_threshold == 70.0

GradeSettings(api_key="test-key", pass_threshold=95.0)  # should raise</pre>
</div>

## 6. Typing a DS Pipeline Config

A real DS pipeline has more than one config object. The pattern is to compose small focused models into a root config that is loaded once:

In [ ]:
from typing import Annotated

from pydantic import BaseModel, Field, model_validator
from pydantic_settings import BaseSettings, SettingsConfigDict


class WeightConfig(BaseModel):
    midterm: Annotated[float, Field(gt=0.0, lt=1.0)] = 0.30
    final: Annotated[float, Field(gt=0.0, lt=1.0)] = 0.45
    project: Annotated[float, Field(gt=0.0, lt=1.0)] = 0.25

    @model_validator(mode="after")
    def weights_sum_to_one(self) -> "WeightConfig":
        total = self.midterm + self.final + self.project
        if abs(total - 1.0) > 1e-6:
            raise ValueError(f"Weights must sum to 1.0, got {total:.4f}")
        return self


class PipelineConfig(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env", env_prefix="GRADE_")

    weights: WeightConfig = WeightConfig()
    pass_threshold: Annotated[float, Field(ge=50.0, le=80.0)] = 60.0
    data_path: str = "data/university_analytics.csv"
    output_path: str = "data/results.parquet"
    debug: bool = False

In [ ]:
config = PipelineConfig()


def compute_grade(midterm: float, final: float, project: float, cfg: PipelineConfig) -> float:
    w = cfg.weights
    return midterm * w.midterm + final * w.final + project * w.project


print(compute_grade(78.0, 82.0, 91.0, config))

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 5 - Nested Config</span><br><br>
<b>Goal:</b> Create a <code>PipelineConfig</code> object. Confirm that <code>config.weights.midterm</code> returns the expected default. Then create one with custom weights <code>WeightConfig(midterm=0.25, final=0.50, project=0.25)</code> and confirm it validates correctly. Finally confirm that invalid nested weights (sum != 1.0) raise a <code>ValidationError</code> on the nested model.
</div>

## 7. Validating a Batch of Records

In DS, data comes in batches. Pydantic validates each record independently; the question is how to handle errors without stopping on the first failure:

In [ ]:
import pandas as pd
from pydantic import ValidationError


def validate_batch(
    records: list[dict],
) -> tuple[list[StudentRecord], list[dict]]:
    valid: list[StudentRecord] = []
    errors: list[dict] = []

    for raw in records:
        try:
            valid.append(StudentRecord.model_validate(raw))
        except ValidationError as e:
            errors.append({**raw, "errors": e.errors()})

    return valid, errors


# Simulate a small batch
sample = [
    {"student_id": "S0001", "midterm_score": 78.5, "final_score": 82.0, "project_score": 90.0, "program": "CS"},
    {
        "student_id": "S0002",
        "midterm_score": 150.0,
        "final_score": 80.0,
        "project_score": 75.0,
        "program": "EE",
    },  # invalid
    {
        "student_id": "INVALID",
        "midterm_score": 70.0,
        "final_score": 68.0,
        "project_score": 72.0,
        "program": "CS",
    },  # invalid
]

valid_records, error_records = validate_batch(sample)
print(f"Valid: {len(valid_records)}, Errors: {len(error_records)}")
for rec in error_records:
    print(rec["student_id"], "->", rec["errors"][0]["msg"])

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Use <code>TypeAdapter</code> to validate a list without a wrapper model</span><br><br>
Pydantic v2's <code>TypeAdapter</code> validates arbitrary types, including <code>list[StudentRecord]</code>, without defining a wrapper model:
<pre style='background:#F4F5F6;padding:10px;border-radius:4px;font-size:0.9em'>from pydantic import TypeAdapter

adapter = TypeAdapter(list[StudentRecord])

# Raises ValidationError for the first invalid record
all_records = adapter.validate_python(df.to_dict(orient="records"))

# Generate JSON Schema for documentation
print(adapter.json_schema())</pre>
This is cleaner than wrapping in a container model when you only need batch validation.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 6 - Batch Validation Report</span><br><br>
<b>Goal:</b> Load <code>university_analytics.csv</code>. Introduce two invalid rows: one with <code>midterm_score=150.0</code> and one with <code>student_id="INVALID"</code>. Run <code>validate_batch()</code> on all rows. Print the count of valid and invalid records, and the error details for the two invalid rows.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>df_with_errors = df.copy()
df_with_errors.loc[0, "midterm_score"] = 150.0
df_with_errors.loc[1, "student_id"] = "INVALID"

valid, errors = validate_batch(df_with_errors.to_dict(orient="records"))
print(f"Valid: {len(valid)}, Errors: {len(errors)}")</pre>
</div>

## Capstone: Typed grade-predictor Pipeline

Bring Pydantic and typer into the `grade-predictor` project end to end.

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Capstone - A Validated, CLI-Driven Pipeline</span><br><br>
<ol>
<li>Define <code>StudentRecord</code> in <code>grade_predictor/models.py</code> with all CSV fields, appropriate constraints, and a <code>student_id</code> format validator</li>
<li>Define <code>PipelineConfig</code> in <code>grade_predictor/config.py</code> with <code>WeightConfig</code>, <code>pass_threshold</code>, and <code>data_path</code></li>
<li>Update <code>compute_grade</code> in <code>core.py</code> to accept a <code>PipelineConfig</code> instead of separate weight arguments</li>
<li>Write a <code>load_and_validate(path: str) -> tuple[list[StudentRecord], list[dict]]</code> function that reads the CSV and validates every row</li>
<li>Add a typer CLI command <code>run</code> that accepts <code>--data-path</code> and <code>--pass-threshold</code>, constructs a <code>PipelineConfig</code>, calls <code>load_and_validate</code>, and prints the valid/error counts</li>
<li>Write two tests: one confirming a valid <code>StudentRecord</code> is constructed correctly, and one confirming an invalid record raises <code>ValidationError</code> with the right field name</li>
</ol>
</div>

## Further Reading

| Resource | Why it matters |
| --- | --- |
| [Pydantic v2 documentation](https://docs.pydantic.dev/latest/) | The primary reference; the migration guide from v1 is worth reading if you encounter older Pydantic code |
| [pydantic-settings documentation](https://docs.pydantic.dev/latest/concepts/pydantic_settings/) | `BaseSettings`, env file loading, nested settings, and secrets |
| [typer documentation](https://typer.tiangolo.com/) | Full reference for CLI commands, subcommands, arguments vs options, and testing |
| [Pydantic v2 validators](https://docs.pydantic.dev/latest/concepts/validators/) | `field_validator`, `model_validator`, `Annotated` constraints |
| [FastAPI + Pydantic](https://fastapi.tiangolo.com/tutorial/body/) | FastAPI uses Pydantic models for request/response; the same `BaseModel` patterns apply directly |
| [pandera](https://pandera.readthedocs.io/) | Schema-level DataFrame validation: the DataFrame equivalent of `StudentRecord` for column dtypes and constraints. Covered in Part 20. |

## Summary

| Concept | Key rule |
| --- | --- |
| `BaseModel` | Validates on construction; coerces compatible types; rejects the rest with `ValidationError` |
| `Field(ge=0, le=100)` | Inline constraints via `Annotated`; replaces manual range checks inside functions |
| `@field_validator` | Single-field business rules; runs after type coercion by default |
| `@model_validator(mode="after")` | Cross-field rules; has access to the fully constructed model |
| `model_dump()` | Convert model to dict for JSON serialisation or pandas |
| `model_validate(dict)` | Construct and validate from a plain dict or JSON string |
| `argparse` | Stdlib CLI parser; verbose but zero-dependency |
| `typer` | Type-annotation-driven CLI; auto-generates help; preferred for DS/MLOps tools |
| `BaseSettings` | Reads from env vars and `.env` files; validates at construction time |
| `env_prefix` | Namespaces all env vars for a settings class |
| Batch validation | Loop with `try/except ValidationError`; collect errors without stopping |
| `TypeAdapter` | Validate arbitrary types including `list[Model]` without a wrapper model |

**Next:** Part 20 covers DataFrame schema validation with Pandera: the same "validate at the boundary" principle, applied to tabular data instead of individual records.